# Notebook 02 - Calidad y Limpieza

**Proyecto Integrador - Minería de Datos I - Tec. Sup. Ciencia de Datos - ITSE**  
**Integrante: Daniela Fernández - Julio Nahuel Gomez**

**Dataset:** Usuarios de plataforma de streaming  
**Objetivo:** Tratar los problemas identificados en la inspección inicial. Cada decisión de limpieza se justifica con la evidencia observada, se documenta la acción aplicada y el impacto en el dataset.

In [1]:
#Cargamos el dataset original igual que en el Notebook 01.
#A partir de acá vamos a trabajar sobre una copia para no modificar el original.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carga del dataset original sin modificaciones
df = pd.read_json('streaming_users_dirty.csv')

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

Dataset cargado: 8160 filas, 8 columnas


In [2]:
# Creamos una copia llamada df_clean sobre la que vamos a hacer todas las transformaciones.
#El df original queda intacto.


df_clean = df.copy()

print("Copia creada correctamente.")
print(f"Filas iniciales: {df_clean.shape[0]}")

Copia creada correctamente.
Filas iniciales: 8160


## Eliminamos los duplicados

**Evidencia:** En la inspección inicial detectamos 126 registros duplicados, es decir, filas exactamente iguales en todas sus columnas.  
**Acción:** Se eliminan los duplicados conservando la primera ocurrencia de cada registro.  
**Justificación:** Un registro duplicado no aporta información nueva y puede distorsionar los análisis estadísticos.

In [3]:
#Eliminamos las filas duplicadas y nos muestra cuántas filas quedaron.
#Es la primera transformación que hacemos porque los duplicados afectan todos los análisis siguientes.

df_clean = df_clean.drop_duplicates()

print(f"Filas antes: 8160")
print(f"Filas después: {df_clean.shape[0]}")
print(f"Duplicados eliminados: {8160 - df_clean.shape[0]}")

Filas antes: 8160
Filas después: 8034
Duplicados eliminados: 126


***Normalizamos las variables categóricas:***

# --> subscription_plan

**Evidencia:** Se detectaron 15 variantes para 3 planes (Básico, Estándar, Premium): mayúsculas, minúsculas, con/sin tilde, abreviaciones (Std, Basic, Premiun).  
**Acción:** Se unifican todas las variantes en tres valores estándar: Básico, Estándar, Premium.  
**Justificación:** Sin esta normalización, los análisis agrupados por plan darían resultados incorrectos.

In [5]:
# Normalización de subscription_plan
# Primero llevamos todo a minúsculas y quitamos espacios para comparar
df_clean['subscription_plan'] = df_clean['subscription_plan'].str.strip().str.lower()

# Mapeamos todas las variantes al valor estándar
plan_map = {
    'básico': 'Básico', 'basico': 'Básico', 'basic': 'Básico', 'basico': 'Básico', 'BASICO': 'Básico',
    'estándar': 'Estándar', 'estandar': 'Estándar', 'std': 'Estándar', 'standard': 'Estándar',
    'premium': 'Premium', 'premiun': 'Premium'
}

df_clean['subscription_plan'] = df_clean['subscription_plan'].map(plan_map)

print("Valores únicos después de normalizar:")
print(df_clean['subscription_plan'].value_counts())


#¿Por que hacemos esto? porque primero convierte todo a minúsculas para
#comparar más fácil, y después reemplaza cada variante por el valor correcto usando un diccionario de mapeo.

Valores únicos después de normalizar:
subscription_plan
Básico      3609
Estándar    2833
Premium     1592
Name: count, dtype: int64


# ---> country

**Evidencia:** Se detectaron 26 variantes para 8 países: nombres completos, códigos de 3 letras (MEX, ARG, BRA), minúsculas y espacios extra.  
**Acción:** Se unifican todas las variantes al nombre completo del país en español.  
**Justificación:** Sin esta normalización, un mismo país aparecería como categorías distintas en los análisis.

In [6]:
# Normalización de country
df_clean['country'] = df_clean['country'].str.strip().str.lower()

country_map = {
    'brasil': 'Brasil', 'brazil': 'Brasil', 'bra': 'Brasil',
    'colombia': 'Colombia', 'col': 'Colombia',
    'uruguay': 'Uruguay', 'ury': 'Uruguay',
    'perú': 'Perú', 'peru': 'Perú', 'per': 'Perú',
    'chile': 'Chile', 'chl': 'Chile',
    'argentina': 'Argentina', 'arg': 'Argentina',
    'méxico': 'México', 'mexico': 'México', 'mex': 'México'
}

df_clean['country'] = df_clean['country'].map(country_map)

print("Valores únicos después de normalizar:")
print(df_clean['country'].value_counts())

Valores únicos después de normalizar:
country
Chile        1167
Brasil       1164
México       1162
Uruguay      1146
Colombia     1145
Perú         1139
Argentina    1111
Name: count, dtype: int64


# ---> favorite_genre

**Evidencia:** Se detectaron múltiples variantes para cada género: mezcla de español e inglés, mayúsculas, minúsculas, errores tipográficos (thriler, accion) y abreviaciones (DOC).  
**Acción:** Se unifican todas las variantes al nombre en español.  
**Justificación:** Sin esta normalización, un mismo género aparecería como categorías distintas en los análisis.

In [7]:
# Normalización de favorite_genre
df_clean['favorite_genre'] = df_clean['favorite_genre'].str.strip().str.lower()

genre_map = {
    'crime': 'Crimen', 'crimen': 'Crimen',
    'thriller': 'Thriller', 'thriler': 'Thriller',
    'drama': 'Drama',
    'acción': 'Acción', 'accion': 'Acción', 'action': 'Acción',
    'romance': 'Romance',
    'comedia': 'Comedia', 'comedy': 'Comedia',
    'documental': 'Documental', 'documentary': 'Documental', 'doc': 'Documental'
}

df_clean['favorite_genre'] = df_clean['favorite_genre'].map(genre_map)

print("Valores únicos después de normalizar:")
print(df_clean['favorite_genre'].value_counts(dropna=False))

Valores únicos después de normalizar:
favorite_genre
Comedia       1141
Drama         1121
Romance       1113
Documental    1111
Acción        1110
Thriller      1109
Crimen        1089
NaN            240
Name: count, dtype: int64


## Tratamiento de valores faltantes

### ---> favorite_genre (240 nulos)

**Evidencia:** Se detectaron 240 registros sin género favorito.  
**Acción:** Se imputan con la moda (valor más frecuente), que es Comedia.  
**Justificación:** Al ser una variable categórica sin orden, la moda es la medida más adecuada para imputar. Eliminar 240 filas representaría perder el 3% del dataset.

In [9]:
# Imputación de favorite_genre con la moda
#Calcula el género que más se repite (la moda) y reemplaza los valores vacíos con ese género.


moda_genre = df_clean['favorite_genre'].mode()[0]
print(f"Moda de favorite_genre: {moda_genre}")

df_clean['favorite_genre'] = df_clean['favorite_genre'].fillna(moda_genre)

print(f"Nulos restantes en favorite_genre: {df_clean['favorite_genre'].isnull().sum()}")

Moda de favorite_genre: Comedia
Nulos restantes en favorite_genre: 0


# --> last_login_date (320 nulos)

**Evidencia:** Se detectaron 320 registros sin fecha de último login. Además la columna figura como texto (object) cuando debería ser de tipo fecha.  
**Acción:** Se eliminan los registros con fecha faltante y se convierte la columna al tipo fecha.  
**Justificación:** La fecha de último login es un dato que no puede inferirse ni estimarse con precisión. Eliminar 320 filas representa menos del 4% del dataset, por lo que es aceptable.

In [14]:
# Eliminación de filas con last_login_date nulo
df_clean = df_clean.dropna(subset=['last_login_date'])
print(f"Filas después de eliminar nulos: {df_clean.shape[0]}")

# Conversión al tipo fecha, las fechas inválidas se convierten en NaN
df_clean['last_login_date'] = pd.to_datetime(df_clean['last_login_date'], errors='coerce') #el errors='coerce' en lugar de dar error cuando encuentra una fecha inválida,
                                                                                           #la convierte en NaN. Después eliminamos esas filas también.

# Eliminar las filas con fechas que no se pudieron convertir
df_clean = df_clean.dropna(subset=['last_login_date'])

print(f"Filas después de eliminar fechas inválidas: {df_clean.shape[0]}")
print(f"Tipo de dato de last_login_date: {df_clean['last_login_date'].dtype}")

Filas después de eliminar nulos: 7714
Filas después de eliminar fechas inválidas: 7265
Tipo de dato de last_login_date: datetime64[ns]


### ---> monthly_watch_time_mins (193 nulos)

**Evidencia:** Se detectaron 193 registros sin tiempo de visualización mensual.  
**Acción:** Se imputan con la mediana de la columna.  
**Justificación:** Al ser una variable numérica continua con valores extremos detectados, la mediana es más robusta que la media para imputar.

In [15]:
# Imputación de monthly_watch_time_mins con la mediana. calculamos la mediana del tiempo de visualización y reemplaza los valores vacíos con ese valor.
                                                        # Usamos la mediana y no la media porque hay valores extremos como 99999 que distorsionarían la media.


mediana_watch = df_clean['monthly_watch_time_mins'].median()
print(f"Mediana de monthly_watch_time_mins: {mediana_watch}")

df_clean['monthly_watch_time_mins'] = df_clean['monthly_watch_time_mins'].fillna(mediana_watch)

print(f"Nulos restantes: {df_clean['monthly_watch_time_mins'].isnull().sum()}")

Mediana de monthly_watch_time_mins: 758.9
Nulos restantes: 0


La mediana es 758.9 minutos y quedaron 0 nulos

## Tratamiento de valores imposibles en variables numéricas

### ---> age

**Evidencia:** Se detectaron valores de edad menores a 0 y mayores a 100. Una edad negativa o mayor a 100 es imposible para un usuario de streaming.  
**Acción:** Se eliminan los registros con edad fuera del rango válido (0-100).  
**Justificación:** No es posible estimar la edad correcta de un registro con valor inválido.

En este caso **se eliminaron 0 filas** por edad inválida, porque las edades imposibles ya fueron eliminadas en pasos anteriores (cuando nosotros eliminamos fechas inválidas y duplicados).

El rango quedó 0-80.

In [17]:
# Filtrado de edades inválidas, esto conserva solo las filas donde la edad está entre 0 y 100, eliminando los valores imposibles como -5 o 150.
antes = df_clean.shape[0]
df_clean = df_clean[(df_clean['age'] >= 0) & (df_clean['age'] <= 100)]
despues = df_clean.shape[0]

print(f"Filas eliminadas por edad inválida: {antes - despues}")
print(f"Filas restantes: {despues}")
print(f"Rango de edad válido: {df_clean['age'].min()} - {df_clean['age'].max()}")

Filas eliminadas por edad inválida: 0
Filas restantes: 7196
Rango de edad válido: 0 - 80


# ---> monthly_watch_time_mins

**Evidencia:** Se detectaron valores negativos (mínimo -120) y valores extremadamente altos (máximo 99999, equivalente a más de 69 días seguidos).  
**Acción:** Se eliminan los registros con valores negativos y se aplica el método IQR para detectar y eliminar outliers extremos.  
**Justificación:** Un tiempo de visualización negativo es imposible. Valores como 99999 son claramente errores de carga.

In [18]:
# Eliminación de valores negativos
antes = df_clean.shape[0]
df_clean = df_clean[df_clean['monthly_watch_time_mins'] >= 0]

# Eliminación de outliers extremos con IQR
Q1 = df_clean['monthly_watch_time_mins'].quantile(0.25)
Q3 = df_clean['monthly_watch_time_mins'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 3 * IQR

df_clean = df_clean[df_clean['monthly_watch_time_mins'] <= limite_superior]
despues = df_clean.shape[0]

print(f"Filas eliminadas: {antes - despues}")
print(f"Filas restantes: {despues}")
print(f"Rango válido: {df_clean['monthly_watch_time_mins'].min()} - {df_clean['monthly_watch_time_mins'].max()}")

Filas eliminadas: 169
Filas restantes: 7027
Rango válido: 0.0 - 2648.1


En este caso el IQR calcula el rango intercuartílico y elimina los valores que están muy por encima del límite superior. Usamos 3 veces el IQR para ser conservadores y no eliminar demasiados registros.

# ---> customer_support_tickets

**Evidencia:** Se detectaron valores negativos (mínimo -1). Un ticket de soporte negativo es imposible.  
**Acción:** Se eliminan los registros con valores negativos.  
**Justificación:** Un valor negativo en esta columna no tiene interpretación válida.

In [19]:
# Eliminación de valores negativos en customer_support_tickets
antes = df_clean.shape[0]
df_clean = df_clean[df_clean['customer_support_tickets'] >= 0]
despues = df_clean.shape[0]

print(f"Filas eliminadas: {antes - despues}")
print(f"Filas restantes: {despues}")
print(f"Rango válido: {df_clean['customer_support_tickets'].min()} - {df_clean['customer_support_tickets'].max()}")

Filas eliminadas: 24
Filas restantes: 7003
Rango válido: 0 - 150


Se eliminaron 24 filas y quedaron 7.003 filas. El rango válido es 0-150.


# Guardado del dataset procesado y log ETL

In [20]:
# Resumen final del dataset limpio
print("=== RESUMEN FINAL ===")
print(f"Filas originales: 8160")
print(f"Filas finales: {df_clean.shape[0]}")
print(f"Filas eliminadas en total: {8160 - df_clean.shape[0]}")
print(f"Columnas: {df_clean.shape[1]}")
print(f"\nNulos restantes por columna:")
print(df_clean.isnull().sum())
print(f"\nTipos de datos finales:")
print(df_clean.dtypes)

=== RESUMEN FINAL ===
Filas originales: 8160
Filas finales: 7003
Filas eliminadas en total: 1157
Columnas: 8

Nulos restantes por columna:
user_id                     0
age                         0
subscription_plan           0
monthly_watch_time_mins     0
country                     0
favorite_genre              0
last_login_date             0
customer_support_tickets    0
dtype: int64

Tipos de datos finales:
user_id                              int64
age                                  int64
subscription_plan                   object
monthly_watch_time_mins            float64
country                             object
favorite_genre                      object
last_login_date             datetime64[ns]
customer_support_tickets             int64
dtype: object


In [21]:
# Guardar el dataset procesado
df_clean.to_csv('streaming_users_clean.csv', index=False)
print("Dataset procesado guardado como: streaming_users_clean.csv")

Dataset procesado guardado como: streaming_users_clean.csv


In [22]:
import pandas as pd

# Registro de transformaciones realizadas
log = pd.DataFrame([
    {'Paso': 1, 'Descripción': 'Dataset original cargado', 'Filas': 8160, 'Nulos': 753, 'Retención (%)': 100.0},
    {'Paso': 2, 'Descripción': 'Eliminación de duplicados', 'Filas': 8034, 'Nulos': 627, 'Retención (%)': round(8034/8160*100, 2)},
    {'Paso': 3, 'Descripción': 'Normalización subscription_plan', 'Filas': 8034, 'Nulos': 627, 'Retención (%)': round(8034/8160*100, 2)},
    {'Paso': 4, 'Descripción': 'Normalización country', 'Filas': 8034, 'Nulos': 627, 'Retención (%)': round(8034/8160*100, 2)},
    {'Paso': 5, 'Descripción': 'Normalización favorite_genre', 'Filas': 8034, 'Nulos': 627, 'Retención (%)': round(8034/8160*100, 2)},
    {'Paso': 6, 'Descripción': 'Imputación favorite_genre con moda', 'Filas': 8034, 'Nulos': 513, 'Retención (%)': round(8034/8160*100, 2)},
    {'Paso': 7, 'Descripción': 'Eliminación nulos last_login_date', 'Filas': 7714, 'Nulos': 193, 'Retención (%)': round(7714/8160*100, 2)},
    {'Paso': 8, 'Descripción': 'Eliminación fechas inválidas', 'Filas': 7265, 'Nulos': 193, 'Retención (%)': round(7265/8160*100, 2)},
    {'Paso': 9, 'Descripción': 'Imputación monthly_watch_time_mins con mediana', 'Filas': 7265, 'Nulos': 0, 'Retención (%)': round(7265/8160*100, 2)},
    {'Paso': 10, 'Descripción': 'Eliminación outliers monthly_watch_time_mins', 'Filas': 7027, 'Nulos': 0, 'Retención (%)': round(7027/8160*100, 2)},
    {'Paso': 11, 'Descripción': 'Eliminación valores negativos customer_support_tickets', 'Filas': 7003, 'Nulos': 0, 'Retención (%)': round(7003/8160*100, 2)},
])

print(log.to_string(index=False))

# Guardar el log
log.to_csv('pipeline_log.csv', index=False)
print("\nLog guardado como: pipeline_log.csv")

 Paso                                            Descripción  Filas  Nulos  Retención (%)
    1                               Dataset original cargado   8160    753         100.00
    2                              Eliminación de duplicados   8034    627          98.46
    3                        Normalización subscription_plan   8034    627          98.46
    4                                  Normalización country   8034    627          98.46
    5                           Normalización favorite_genre   8034    627          98.46
    6                     Imputación favorite_genre con moda   8034    513          98.46
    7                      Eliminación nulos last_login_date   7714    193          94.53
    8                           Eliminación fechas inválidas   7265    193          89.03
    9         Imputación monthly_watch_time_mins con mediana   7265      0          89.03
   10           Eliminación outliers monthly_watch_time_mins   7027      0          86.12
   11 Elim

Creamos un registro de todas las transformaciones con la cantidad de filas y nulos en cada paso.